# Modular eval + head registry — the plug-in seam
How to run **any** protein-conditioning head through the SAME leakage-controlled, family-disjoint evaluation. Synthetic data — runs from a clone, no lab assets. See `docs/MERGE_PLAN.md` §6 (shared interfaces) and `src/mmpartnet/eval/`.

## 1. The head registry
`RunConfig.conditioning` selects a head by name; `build_head(name)` lazy-imports it. To add your model: drop a module in `models/`, implement a head, add one row in `models/registry.py`.

In [1]:
from mmpartnet.models import list_heads, head_spec, build_head
for n in list_heads():
    s = head_spec(n)
    print(f'{n:14s} task={s.task:7s} inputs={s.inputs:6s} owner={s.owner:9s} {s.cls}')
print()
print('build_head("xattn") ->', build_head('xattn').__module__)

conditioned    task=binary  inputs=pooled owner=ours      ConditionedHead
early          task=binary  inputs=pooled owner=cgerards  EarlyFusion
film           task=profile inputs=perres owner=dgu       ProteinCellFiLMProfileHead
perres         task=profile inputs=perres owner=ours      CrossAttnHead
perres_bidir   task=profile inputs=perres owner=ours      BidirCrossAttnHead
xattn          task=profile inputs=perres owner=dgu       ProteinCellCrossAttentionProfileHead
xattn2         task=profile inputs=perres owner=dfra      ProteinCellCrossAttentionProfileHead

build_head("xattn") -> mmpartnet.models.cross_attention_dgu


## 2. The leakage-controlled leave-out-RBP gate
`held_rbp_gate(score_fn, ...)` is model-agnostic: give a `score_fn(feats, protein_vec)` and the held-out RBPs, get per-RBP AUROC + the mandatory protein-shuffle control, **fire-checked**. A protein-USING head must make the shuffle collapse (real > shuffle); a protein-IGNORING head does not — and the harness FLAGS that (B3 discipline: a silent control is a broken eval, not a win).

In [2]:
import numpy as np
from mmpartnet.eval import held_rbp_gate
rng = np.random.default_rng(0)
K, W = 6, 400                      # 6 held RBPs, 400 windows
t = rng.integers(0, K, size=W)     # each window's latent 'bound-by' RBP
feats = np.eye(K)[t]               # (W,K) window features (one-hot of the latent type)
te_y  = np.eye(K)[t]               # (W,K) labels: window w is bound by RBP t[w]
prot  = {k: np.eye(K)[k] for k in range(K)}   # protein rep = identity of RBP k
ti_keep = list(range(K)); held_k = list(range(K))
syms = [f'RBP{k}' for k in range(K)]; fam = [f'fam{k%3}' for k in range(K)]

protein_using  = lambda F, pv: F @ pv            # uses the protein rep
protein_blind  = lambda F, pv: F.sum(1)          # ignores the protein rep

for name, fn in [('protein-USING', protein_using), ('protein-BLIND', protein_blind)]:
    r = held_rbp_gate(fn, feats, held_k, prot, te_y, ti_keep, syms, fam, bar=0.65)
    ps = r.controls['protein_shuffle']
    print(f'{name:14s} real={r.real_mean:.2f} shuffle={ps["mean_auroc"]:.2f} '
          f'fired={ps["fired"]} warn={ps["warn"]}  honest_zero_shot={r.honest_zero_shot}')

protein-USING  real=1.00 shuffle=0.40 fired=True warn=False  honest_zero_shot=False
protein-BLIND  real=0.50 shuffle=0.50 fired=False warn=True  honest_zero_shot=False


**Read-out:** the protein-USING head separates the RBPs (real AUROC ~1.0) and the protein-shuffle collapses it (control *fires*); the protein-BLIND head scores identically under shuffle, so the control does NOT fire and is flagged. `honest_zero_shot=False` because the default checkpoint is the leaked all-223 body — the number is proxy-level until leave-out weights are set.

## 3. The family-disjoint guarantee + splits
`family_disjoint_assert` refuses a split where a family appears in both train and eval; the `paralog` axis is the within-well transfer control (leave-out-chromosome = the `naive` axis).

In [3]:
from mmpartnet.eval import family_disjoint_assert
from mmpartnet.splits.registry import list_splits, get_split
from mmpartnet.splits.base import SplitConfig
print('split axes:', list_splits())
sp = get_split(['A','B','C','D'], SplitConfig(axis='paralog'),
               meta={'paralog': {'A':'g1','B':'g1','C':'g2','D':'g2'}})
print('paralog leave-out:', 'train', sp.train, 'test', sp.test)
try:
    family_disjoint_assert(['A','B'], ['C','A'], {'A':'f1','B':'f2','C':'f1'})
except AssertionError as e:
    print('caught:', str(e)[:70], '...')

split axes: ['family', 'naive', 'paralog', 'rbp_holdout']
paralog leave-out: train ('C', 'D') test ('A', 'B')
caught: family leakage: 1 families in BOTH train and eval (['f1']); use a fami ...


## 4. CORAL-comparable metrics + the affinity certificate
`coral_f1_auroc(..., seen_mask=)` always reports the UNSEEN (cold-start) block — the real claim. `validate_grid` is the affinity far>near test with a family-block permutation null.

In [4]:
from mmpartnet.metrics import coral_f1_auroc, validate_grid
pred = np.array([0.9,0.8,0.7,0.2,0.1,0.05]); y = np.array([1,1,1,0,0,0])
seen = np.array([1,1,0,1,0,0], bool)
r = coral_f1_auroc(pred, y, seen_mask=seen, best_thr=True)
print('overall', r['overall']['f1'], r['overall']['auroc'], '| unseen n', r['unseen']['n'])
vg = validate_grid(pred, np.array([.1,.2,.3,.9,1.,1.1]), n_perm=500, seed=0)
print('affinity far>near effect', round(vg['effect'],3), 'p', round(vg['p'],4))

overall 1.0 1.0 | unseen n 3
affinity far>near effect 0.775 p 0.01


## 5. Plug YOUR head in (recipe)
```python
# 1) models/your_head.py: implement forward(rna_feats, protein_rep) -> logit and/or profile
# 2) models/registry.py: add HeadSpec('mine','your_head','YourHead','profile','perres','you', '...')
# 3) RunConfig(conditioning='mine'); build_head('mine')(**dims)
# 4) evaluate: held_rbp_gate(torch_head_scorer(head), feats, held_k, prot, te_y, ti_keep, syms, fam)
#    -> same controls + family-disjoint guarantee everyone else is held to.
```
Set `ML4RG_PARNET_WEIGHTS` to a leave-out checkpoint (+ `ML4RG_HONEST_ZEROSHOT=1`) to promote a run from proxy to a real held-out claim.